# Étape 3 — Interroger les LLMs
Ce notebook lit les prompts générés dans `data/prompts.csv`, interroge chaque modèle via l'API Groq, et enregistre les réponses dans `data/responses.csv`.

**Il peut être interrompu et relancé** : les paires déjà traitées sont automatiquement ignorées.

In [ ]:
import csv
import os
import time
import uuid
import requests
from pathlib import Path

PROJECT_DIR    = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR       = PROJECT_DIR / 'data'
PROMPTS_FILE   = DATA_DIR / 'prompts.csv'
RESPONSES_FILE = DATA_DIR / 'responses.csv'

print('Projet      :', PROJECT_DIR)
print('Prompts     :', PROMPTS_FILE.exists())
print('Responses   :', RESPONSES_FILE.exists())

## 1. Configuration

In [ ]:
# Clé API Groq — mets ta clé ici ou dans une variable d'environnement GROQ_API_KEY
API_KEY = os.getenv('GROQ_API_KEY', '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')

MODELES = [
    'llama-3.1-8b-instant',
    'openai/gpt-oss-20b',
]

TEMPERATURE = 1.0
MAX_TOKENS  = 600
PAUSE       = 1   # secondes entre chaque appel

print('Modèles configurés :', MODELES)

## 2. Chargement des prompts et des réponses déjà enregistrées

In [ ]:
with open(PROMPTS_FILE, encoding='utf-8') as f:
    prompts = list(csv.DictReader(f))
print(f'{len(prompts)} prompts chargés')

# Réponses déjà faites (pour reprise)
deja_faits = set()
FIELDNAMES = ['response_id', 'prompt_id', 'model', 'temperature', 'response_text']

if RESPONSES_FILE.exists() and RESPONSES_FILE.stat().st_size > 0:
    with open(RESPONSES_FILE, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            deja_faits.add((row['prompt_id'], row['model']))

total    = len(prompts) * len(MODELES)
restants = total - len(deja_faits)
print(f'{len(deja_faits)} réponses déjà enregistrées')
print(f'{restants} appels API restants sur {total} au total')

## 3. Interrogation des modèles

In [ ]:
def interroger_ia(prompt_text, modele):
    while True:
        r = requests.post(
            'https://api.groq.com/openai/v1/chat/completions',
            headers={'Authorization': f'Bearer {API_KEY}'},
            json={
                'model': modele,
                'messages': [{'role': 'user', 'content': prompt_text}],
                'max_tokens': MAX_TOKENS,
                'temperature': TEMPERATURE,
            },
        )
        data = r.json()
        if 'choices' in data:
            return data['choices'][0]['message']['content']
        if r.status_code == 429:
            print('   ... limite atteinte, pause 2s puis on réessaie')
            time.sleep(2)
            continue
        print('\n ERREUR API :', data)
        raise RuntimeError('Erreur API — voir message ci-dessus.')


mode = 'a' if RESPONSES_FILE.exists() and RESPONSES_FILE.stat().st_size > 0 else 'w'
nouveaux = 0

with open(RESPONSES_FILE, mode, newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    if mode == 'w':
        writer.writeheader()

    compteur = len(deja_faits)
    for modele in MODELES:
        print(f'\n=== Modèle : {modele} ===')
        for p in prompts:
            if (p['prompt_id'], modele) in deja_faits:
                continue
            compteur += 1
            print(f'[{compteur}/{total}] {p["prompt_text"][:70]}...')
            reponse = interroger_ia(p['prompt_text'], modele)
            writer.writerow({
                'response_id':   str(uuid.uuid4()),
                'prompt_id':     p['prompt_id'],
                'model':         modele,
                'temperature':   TEMPERATURE,
                'response_text': reponse,
            })
            f.flush()
            nouveaux += 1
            time.sleep(PAUSE)

print(f'\n✓ {nouveaux} nouvelles réponses ajoutées dans {RESPONSES_FILE}')
print(f'  Total : {len(deja_faits) + nouveaux} / {total}')